# Lab: shortest path to a Ground Station
In about 10 minutes and with minimal configuration, you will:
1. spin up a small constellation from a plain configuration (like in [create_my_first_simulation.ipynb](create_my_first_simulation.ipynb))
2. add a **Ground Station** and a **User Terminal** by hand, directly through `SimulationManager` (like in [deeply_understand_sat_com_topology.ipynb](deeply_understand_sat_com_topology.ipynb))
3. run the simulation for 10 simulated minutes
4. at every step, compute the shortest path between the User Terminal and the Ground Station with `networkx.shortest_path`

By the end you will have a routing path such as:
```
UserTerminal#1 -> Satellite#23 -> Satellite#31 -> Satellite#7 -> GroundStation#1
```
recomputed every minute as the satellites move.

In [ ]:
import networkx as nx

from sat_com_adapter.networkx_builder.networkx_builder import NetworkxBuilder
from sat_com_builder.configuration_manager import BaseConfigurationManager
from sat_com_builder.models import SimulationProperty
from sat_com_model.models import create_ground_station, create_user_terminal, GroundObjectDomain


## 1. A minimal constellation
We only need a `walker_shells` entry to get satellites and inter-satellite links. We deliberately leave `ground_objects_properties` empty: the Ground Station and the User Terminal will be added by hand in the next step.

The shell below is intentionally small (48 satellites) so the lab runs fast, while still being dense enough to always have a satellite in view of both ground objects.

In [ ]:
lab_config = {
    "simulation_name": "Shortest Path Lab",
    "start_date": "2026-01-01 00:00:00.000000",
    "end_date": "2026-01-01 00:10:00.000000",
    "movement_model": "pyorbital",
    "distance_model": "sklearn",
    "ground_objects_properties": [],
    "walker_shells": [
        {
            "type": "delta",
            "constellation_property": {
                "identifier": "Lab Walker",
                "amount_of_orbit_plane": 6,
                "amount_of_satellite_per_orbit_plane": 8,
                "inclination": 53.0,
                "phase_difference_between_satellites": True,
                "mean_revolution_per_day": 15.05,
            },
            "orbital_connectivity_property": {
                "adjacent_inter_satellite_shifting": 0,
                "maximum_inter_satellite_count": 4,
                "maximum_inter_satellite_range_distance": 6000,
                "maximum_ground_station_range": 2000,
                "maximum_user_terminal_range": 2000,
                "maximum_connected_ground_object": 10000,
                "maximum_connected_user_terminal": 1000,
                "maximum_connected_ground_station": 10,
            },
            "ground_object_white_list": [],
        }
    ],
}

simulation_properties = SimulationProperty(**lab_config)
configuration_manager = BaseConfigurationManager(simulation_property=simulation_properties)

simulation_manager = configuration_manager.load_simulation()

print(f"{len(simulation_manager.get_satellites())} satellites loaded, 0 ground object (added manually next)")


## 2. Add the Ground Station and the User Terminal manually
`SimulationManager` exposes `add_ground_station` and `add_user_terminals` to register ground objects one by one, without any configuration file.

To let the simulation automatically keep them connected to the best visible satellite while it ticks, each ground object needs a `GroundObjectDomain` describing its connection strategy. We use `best-angle-until-disconnection`: stay connected to the same satellite as long as it is visible, then switch to the best newly visible one. This is the same mechanism used by the configuration-driven notebooks, only wired by hand here.

In [ ]:
ground_station_domain = GroundObjectDomain(
    identifier="manual-ground-station",
    type="ground_station",
    elevation_above_horizon=20,
    maximum_satellite_range_distance=2000,
    ground_to_space_connections_strategy="best-angle-until-disconnection",
)
simulation_manager.add_ground_object_domain(ground_station_domain)

ground_station = create_ground_station(object_id=1, domain=ground_station_domain.identifier)
ground_station.label = "Paris"
ground_station.set_position(longitude=2.349014, latitude=48.864716, altitude=0)
ground_station.ground_object_domain = ground_station_domain

simulation_manager.add_ground_station(ground_station)


In [ ]:
user_terminal_domain = GroundObjectDomain(
    identifier="manual-user-terminal",
    type="user_terminal",
    elevation_above_horizon=20,
    maximum_satellite_range_distance=2000,
    ground_to_space_connections_strategy="best-angle-until-disconnection",
)
simulation_manager.add_ground_object_domain(user_terminal_domain)

user_terminal = create_user_terminal(object_id=1, domain=user_terminal_domain.identifier)
user_terminal.label = "Toulouse"
user_terminal.set_position(longitude=1.444000, latitude=43.604500, altitude=0)
user_terminal.ground_object_domain = user_terminal_domain

simulation_manager.add_user_terminals(user_terminal)


## 3. Compute the shortest path with `networkx`
At every simulation tick we:
1. rebuild the NetworkX graph of the current topology (satellites, ground objects and their links)
2. call `networkx.shortest_path` between the User Terminal node and the Ground Station node

We ask for two variants: the path with the **fewest hops** (no `weight` argument, plain BFS) and the **shortest by distance** (`weight="distance"`, Dijkstra), since both are useful depending on what you optimize for.

In [ ]:
shortest_path_history = []


def describe_node(graph, node_id):
    node = graph.nodes[node_id]
    return f"{node['type']}#{node['object_id']}"


def compute_shortest_path():
    graph_builder = NetworkxBuilder(simulation_manager=simulation_manager, export_link_length=True)
    graph_builder.add_topology_objects()
    graph_builder.add_links()
    graph = graph_builder.build()

    source_id = user_terminal.topology_uniq_id
    target_id = ground_station.topology_uniq_id

    current_time = simulation_manager.time_manager.current_time

    try:
        fewest_hops_path = nx.shortest_path(graph, source=source_id, target=target_id)
        shortest_by_distance_path = nx.shortest_path(
            graph, source=source_id, target=target_id, weight="distance"
        )
        distance_km = (
            nx.shortest_path_length(graph, source=source_id, target=target_id, weight="distance")
            / 1000
        )

        readable_path = " -> ".join(describe_node(graph, node_id) for node_id in shortest_by_distance_path)

        print(
            f"[{current_time:%H:%M:%S}] {len(fewest_hops_path) - 1} hop(s), "
            f"{distance_km:.0f} km : {readable_path}"
        )

        shortest_path_history.append(
            {
                "time": current_time,
                "hops": len(fewest_hops_path) - 1,
                "distance_km": distance_km,
                "path": shortest_by_distance_path,
            }
        )
    except nx.NetworkXNoPath:
        print(f"[{current_time:%H:%M:%S}] no path currently available")


simulation_manager.time_manager.register_action(compute_shortest_path)


## 4. Run the simulation for 10 minutes
`tick_until_the_end` advances the clock, which triggers (in order):
1. the default update protocol (satellite positions, then Ground Station / User Terminal links)
2. our `compute_shortest_path` action, registered above

We tick every 60 seconds, for the whole 10 simulated minutes configured in `lab_config`.

In [ ]:
simulation_manager.time_manager.tick_until_the_end(ticking_time_in_seconds=60)


## 5. Recap
A quick summary of what happened over the 10 minutes: how the hop count and the path length evolved as satellites moved in and out of view.

Don't be surprised if a few minutes report "no path currently available": the Ground Station and the User Terminal each independently pick their *own* best satellite (`best-angle-until-disconnection`), so a path only exists when both currently sit under the same satellite, or under two satellites that also happen to share an inter-satellite link. That is expected behavior of this simple strategy, not a bug.

In [ ]:
if shortest_path_history:
    hop_counts = [entry["hops"] for entry in shortest_path_history]
    distances = [entry["distance_km"] for entry in shortest_path_history]

    print(f"{len(shortest_path_history)} snapshot(s) with a valid path")
    print(f"hops   : min={min(hop_counts)}, max={max(hop_counts)}")
    print(f"distance: min={min(distances):.0f} km, max={max(distances):.0f} km")
else:
    print("No path was ever found, try increasing maximum_satellite_range_distance or the shell size")


## What's next?
- Swap `best-angle-until-disconnection` for `everything-visible` on the domains above and see how the path (and its stability) changes.
- Move the Ground Station and the User Terminal further apart and observe how the hop count grows.
- Head back to [create_my_first_simulation.ipynb](create_my_first_simulation.ipynb) to see how the same Ground Station / User Terminal setup can be described declaratively for a whole fleet of ground objects instead of one at a time.